Setup and load

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42

clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")
display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [2]:
print("Classification:", clf_df.shape, "| Regression:", reg_df.shape)
display(clf_df.isna().sum().to_frame("missing (clf)"), reg_df.isna().sum().to_frame("missing (reg)"))
print(clf_df["escalated"].value_counts().to_dict(), "| prevalence:", clf_df["escalated"].mean().round(4))

Classification: (900, 7) | Regression: (800, 7)


,missing (clf)
ticket_id,0
complexity_score,22
team_experience_years,36
priority,13
channel,9
customer_tenure_months,31
escalated,0


,missing (reg)
ticket_id,0
complexity_score,16
team_experience_years,24
priority,0
channel,0
customer_tenure_months,20
resolution_time_hours,0


{0: 709, 1: 191} | prevalence: 0.2122


## Step 1 – Data overview
- Two datasets: classification (900 tickets, target `escalated`) and regression (800 tickets, target `resolution_time_hours`). Both share the same 5 predictors; `ticket_id` is an identifier and is excluded.
- Missing values appear in the numeric predictors of both files, and also in `priority` and `channel` of the classification file, so imputation is required.
- About 21% of tickets are escalated: the classes are imbalanced, which is why F1 (not accuracy) is the primary metric.

In [3]:
num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])

X = clf_df.drop(columns=["ticket_id", "escalated"])
y = clf_df["escalated"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

print(X_train.shape, X_test.shape)
print("Positives  train:", y_train.sum(), round(y_train.mean(), 4), "| test:", y_test.sum(), round(y_test.mean(), 4))

(675, 5) (225, 5)
Positives  train: 143 0.2119 | test: 48 0.2133


## Step 2 – Experiment A: split and preprocessing
- `ticket_id` is dropped (an identifier, not a predictor). Target = `escalated`; positive class = 1.
- 75/25 **stratified** split: the escalation rate is 21.2% in train and 21.3% in test, so evaluation is on a representative test set with 48 positives. The test set is then locked away.
- Numeric: median imputation (robust to outliers) + standardization (needed by k-NN and helpful for logistic regression). Categorical: most-frequent imputation + one-hot (`handle_unknown="ignore"` so unseen categories don't crash).
- Preprocessing is built by `make_preprocessor()` and placed **inside every Pipeline**, so it is re-fit on the training part of each CV fold only (no leakage).

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidates = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "k-NN (k=5)": KNeighborsClassifier(),
    "Decision tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
}

rows = []
for name, model in candidates.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", model)])
    s = cross_validate(pipe, X_train, y_train, cv=skf,
                       scoring=["f1", "precision", "recall", "accuracy"])
    print(name, "fold F1:", s["test_f1"].round(3))
    rows.append({"model": name, "F1 mean": s["test_f1"].mean(), "F1 std": s["test_f1"].std(),
                 "Precision": s["test_precision"].mean(), "Recall": s["test_recall"].mean(),
                 "Accuracy": s["test_accuracy"].mean()})

cv_table = pd.DataFrame(rows).set_index("model").round(3)
cv_table

Logistic regression fold F1: [0.381 0.348 0.39  0.476 0.316]
k-NN (k=5) fold F1: [0.333 0.273 0.41  0.34  0.19 ]
Decision tree fold F1: [0.36  0.23  0.542 0.423 0.453]


,F1 mean,F1 std,Precision,Recall,Accuracy
model,,,,,
Logistic regression,0.382,0.054,0.624,0.280,0.809
k-NN (k=5),0.309,0.074,0.485,0.231,0.781
Decision tree,0.401,0.104,0.402,0.411,0.739


### Observations (Step 3)
- All models use the **same** 5 stratified folds, so the comparison is fair. F1 is the primary metric because the positive class is rare and both FP and FN matter.
- Untuned decision tree has the highest mean F1 (0.401) but also the largest fold-to-fold std (0.104, fold F1 ranging 0.23–0.54): it is unstable and overfits (deepest, unpruned tree). Logistic regression is slightly lower (0.382) but much steadier (std 0.054). k-NN (k=5) is weakest.
- Accuracy ranks the models differently (logistic regression best at 0.809, tree worst at 0.739): accuracy rewards predicting the majority class, which is why it is not used for selection.

In [5]:
param_grids = {
    "Logistic regression": (LogisticRegression(max_iter=1000), {"model__C": [0.01, 0.1, 1, 10, 100]}),
    "k-NN": (KNeighborsClassifier(), {"model__n_neighbors": [5, 9, 15, 21, 31]}),
    "Decision tree": (DecisionTreeClassifier(random_state=RANDOM_STATE),
                      {"model__max_depth": [2, 3, 4, 5, 6, 8],
                       "model__min_samples_leaf": [1, 5, 10, 20]}),
}

searches = {}
for name, (model, grid) in param_grids.items():
    gs = GridSearchCV(Pipeline([("prep", make_preprocessor()), ("model", model)]),
                      grid, scoring="f1", cv=skf)
    gs.fit(X_train, y_train)
    i = gs.best_index_
    searches[name] = gs
    print(f"{name}: {gs.best_params_} | CV F1 = {gs.best_score_:.3f} +/- {gs.cv_results_['std_test_score'][i]:.3f}")

selected_name = max(searches, key=lambda k: searches[k].best_score_)
print("Selected:", selected_name)

Logistic regression: {'model__C': 10} | CV F1 = 0.395 +/- 0.061
k-NN: {'model__n_neighbors': 5} | CV F1 = 0.309 +/- 0.074
Decision tree: {'model__max_depth': 6, 'model__min_samples_leaf': 20} | CV F1 = 0.435 +/- 0.102
Selected: Decision tree


### Tuning (Step 4)
Search spaces (all with the same stratified 5-fold CV, scoring = F1, on the training set only): logistic regression C ∈ {0.01…100}; k-NN k ∈ {5, 9, 15, 21, 31}; decision tree max_depth ∈ {2,3,4,5,6,8} × min_samples_leaf ∈ {1,5,10,20}.
- Limiting tree depth and requiring ≥20 samples per leaf lifted CV F1 from 0.401 to **0.435** and acts as regularization.
- The tree has the best CV F1, but its std (0.10) is about twice that of logistic regression (0.395 ± 0.061); the gap (0.04) is smaller than the fold spread, so the two are close.
- Selection rule fixed in advance: highest tuned CV F1 → decision tree. `GridSearchCV` refits the winner on the full training set.

In [6]:
final_clf = searches[selected_name].best_estimator_
y_pred = final_clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(cm)
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(classification_report(y_test, y_pred, target_names=["not escalated", "escalated"], digits=3))

[[165  12]
 [ 36  12]]
TN=165  FP=12  FN=36  TP=12
               precision    recall  f1-score   support

not escalated      0.821     0.932     0.873       177
    escalated      0.500     0.250     0.333        48

     accuracy                          0.787       225
    macro avg      0.660     0.591     0.603       225
 weighted avg      0.752     0.787     0.758       225



### Final classification test result (Step 5, evaluated once)
Decision tree (max_depth=6, min_samples_leaf=20): precision 0.50, recall 0.25, **F1 0.333**, accuracy 0.787.
- **False positives (12):** tickets flagged for escalation that did not need it: wasted review effort, mild cost.
- **False negatives (36):** real escalations we missed; this is 75% of the 48 escalated tickets and is the serious error if late escalation is costly.
- Test F1 (0.333) is well below the CV F1 (0.435). Choosing the best of 12+ settings by CV makes the CV score optimistic, and with only 48 positives the test estimate is noisy too. Neither number is more than a rough guide.
- A threshold below 0.5 would raise recall at the price of more FPs. That choice should follow the real costs of FN vs FP.

In [7]:
Xr = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
yr = reg_df["resolution_time_hours"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.25, random_state=RANDOM_STATE)
print(Xr_train.shape, Xr_test.shape)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
reg_candidates = {
    "Linear regression": LinearRegression(),
    "Ridge (alpha=1)": Ridge(alpha=1.0),
    "Decision tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
}

rows = []
for name, model in reg_candidates.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", model)])
    s = cross_validate(pipe, Xr_train, yr_train, cv=kf,
                       scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error"])
    rows.append({"model": name,
                 "MAE mean": -s["test_neg_mean_absolute_error"].mean(),
                 "MAE std": s["test_neg_mean_absolute_error"].std(),
                 "RMSE mean": -s["test_neg_root_mean_squared_error"].mean(),
                 "RMSE std": s["test_neg_root_mean_squared_error"].std()})
reg_cv_table = pd.DataFrame(rows).set_index("model").round(3)
reg_cv_table

(600, 5) (200, 5)


,MAE mean,MAE std,RMSE mean,RMSE std
model,,,,
Linear regression,2.034,0.096,2.989,0.427
Ridge (alpha=1),2.035,0.095,2.989,0.427
Decision tree,3.145,0.341,4.615,0.562


## Step 6 – Experiment B: regression
- Target `resolution_time_hours`; `ticket_id` excluded; plain 75/25 split (no stratification for a continuous target); same preprocessing pipeline; `KFold(5, shuffle=True)`.
- MAE is the primary metric: it is in hours, easy to explain, and not dominated by a few very long tickets. RMSE is reported alongside for comparison.
- Linear and ridge are essentially identical (2.03 h); the unpruned tree is much worse (3.15 h) and far less stable (MAE std 0.34 vs 0.10), a sign of overfitting.

In [8]:
reg_grids = {
    "Ridge": (Ridge(), {"model__alpha": [0.01, 0.1, 1, 10, 100, 300]}),
    "Decision tree": (DecisionTreeRegressor(random_state=RANDOM_STATE),
                      {"model__max_depth": [2, 3, 4, 5, 6, 8],
                       "model__min_samples_leaf": [1, 5, 10, 20]}),
}

reg_searches = {}
for name, (model, grid) in reg_grids.items():
    gs = GridSearchCV(Pipeline([("prep", make_preprocessor()), ("model", model)]),
                      grid, scoring="neg_mean_absolute_error", cv=kf)
    gs.fit(Xr_train, yr_train)
    i = gs.best_index_
    reg_searches[name] = gs
    print(f"{name}: {gs.best_params_} | CV MAE = {-gs.best_score_:.3f} +/- {gs.cv_results_['std_test_score'][i]:.3f}")

reg_selected = max(reg_searches, key=lambda k: reg_searches[k].best_score_)
print("Selected:", reg_selected)

Ridge: {'model__alpha': 0.01} | CV MAE = 2.034 +/- 0.096
Decision tree: {'model__max_depth': 4, 'model__min_samples_leaf': 5} | CV MAE = 2.395 +/- 0.195
Selected: Ridge


In [9]:
final_reg = reg_searches[reg_selected].best_estimator_
pred = final_reg.predict(Xr_test)

mae = mean_absolute_error(yr_test, pred)
rmse = np.sqrt(mean_squared_error(yr_test, pred))
baseline = mean_absolute_error(yr_test, np.full(len(yr_test), yr_train.median()))
print(f"Test MAE = {mae:.3f} h | Test RMSE = {rmse:.3f} h | Median-baseline MAE = {baseline:.3f} h")

Test MAE = 2.138 h | Test RMSE = 3.274 h | Median-baseline MAE = 3.269 h


### Tuning and final regression result (Step 7)
Search spaces (5-fold CV, scoring = MAE, training set only): ridge alpha ∈ {0.01, 0.1, 1, 10, 100, 300}; tree max_depth ∈ {2,3,4,5,6,8} × min_samples_leaf ∈ {1,5,10,20}.
- Ridge CV MAE = 2.034 ± 0.096 h; best tree (depth 4, leaf 5) = 2.395 ± 0.195 h. Ridge wins and is also more stable.
- The best alpha is the smallest value in the grid, so regularization is not helping: the relationship is close to linear and ridge ≈ ordinary least squares.
- **Test (evaluated once): MAE 2.14 h, RMSE 3.27 h.** Always predicting the training median gives MAE 3.27 h, so the model cuts typical error by about a third.